# 02 - Modeling, Calibration, and Explainability
## Indian Liver Patient Dataset (ILPD)

This narrative notebook walks through model training, repeated cross-validation, threshold tuning,
independent holdout test evaluation, and global/local SHAP explainability.

In [1]:
import joblib
import json
import pandas as pd
import matplotlib.pyplot as plt

from src.config import MODEL_PATH, METRICS_PATH, CV_RESULTS_PATH, FIGURES_DIR
from src.explain import explain_single_patient

### 1. Cross-Validation Results

In [2]:
cv_results = pd.read_csv(CV_RESULTS_PATH)
cv_results

### 2. Independent Test Set Performance & Confidence Intervals

In [3]:
with open(METRICS_PATH, 'r') as f:
    metrics = json.load(f)

print(f"Selected Model: {metrics['model_name']}")
print(f"Test ROC-AUC: {metrics['roc_auc']:.4f}")
print(f"Test Average Precision: {metrics['average_precision']:.4f}")
print(f"Decision Thresholds: t_low = {metrics['thresholds']['t_low']:.4f}, t_high = {metrics['thresholds']['t_high']:.4f}")
print(f"Sensitivity @ t_low: {metrics['metrics_at_t_low']['recall']:.1%}")
print(f"Specificity @ t_low: {metrics['metrics_at_t_low']['specificity']:.1%}")

### 3. Local Feature Attribution (SHAP Example)

In [4]:
artifact = joblib.load(MODEL_PATH)
pipeline = artifact['pipeline']

sample_patient = pd.DataFrame([{
    'age': 55,
    'gender': 'Male',
    'total_bilirubin': 4.5,
    'direct_bilirubin': 2.1,
    'alk_phos': 320,
    'alt': 85,
    'ast': 110,
    'total_protein': 6.2,
    'albumin': 2.8,
    'ag_ratio': 0.8
}])

p_score = pipeline.predict_proba(sample_patient)[0, 1]
print(f"Predicted Patient Risk Score: {p_score:.1%}")

explanations = explain_single_patient(pipeline, sample_patient, top_k=5)
pd.DataFrame(explanations)